# Description

In this notebook, I will explore the benchmark Human Eval and using Llama to generate it.

In [1]:
import os 
import sys
import numpy as np 
import pandas as pd 
import re
import io
import re
import ast
import types
import unittest
import importlib
from typing import List, Tuple, Dict, Any, Set
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import transformers

In [2]:
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="cuda",
)

MAX_NEW_TOKENS = 1024

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

# 1. Load data

In [3]:
PATH_CSV_DATA = "data/raw_data/human_eval.csv"

In [4]:
df = pd.read_csv(PATH_CSV_DATA)
print(f"Dataframe shape: {df.shape}")
df.sample(1)

Dataframe shape: (164, 5)


,task_id,prompt,canonical_solution,test,entry_point
14,HumanEval/14,from typing import List\n\n\ndef all_prefixes(...,result = []\n\n for i in range(len(stri...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da...",all_prefixes


In [5]:
idx = np.random.randint(0, df.shape[0])

code_description = df.loc[idx, "prompt"]
test_case = df.loc[idx, "test"]
entry_point = df.loc[idx, "entry_point"]

print("Code description:")
print(code_description)
print("=" * 20)
print("Test Case:")
print(test_case)

Code description:

def count_upper(s):
    """
    Given a string s, count the number of uppercase vowels in even indices.
    
    For example:
    count_upper('aBCdEf') returns 1
    count_upper('abcdefg') returns 0
    count_upper('dBBE') returns 0
    """

Test Case:
def check(candidate):

    # Check some simple cases
    assert candidate('aBCdEf')  == 1
    assert candidate('abcdefg') == 0
    assert candidate('dBBE') == 0
    assert candidate('B')  == 0
    assert candidate('U')  == 1
    assert candidate('') == 0
    assert candidate('EEEE') == 2

    # Check some edge cases that are easy to work out by hand.
    assert True




# 2. Using Llama to generate sample

## 2.1. Generate code

In [6]:
def extract_function(llm_text):
    # 1) Grab text between <code>...</code>
    m = re.search(r"<code>\s*(.*?)\s*</code>", llm_text, flags=re.S|re.M)
    if not m:
        raise ValueError("No <code> block found")
    code = m.group(1)

    # 2) Optionally, if the model sometimes adds backticks, strip them
    code = re.sub(r"^```(?:python)?\s*|\s*```$", "", code.strip())

    return code

In [7]:
constraints = """
Output only a complete and valid Python code for this function. 
Do not add more explanations or surrounding text and Do not change the provided function signature.
Wrap your output strictly between the markers:
<code>
... your code ...
</code>
"""

input_prompt = f"""write a complete python function
based on the following description:\n{code_description}.\n
with the following constraints:\n{constraints}
"""

print("Input prompt to:")
print(input_prompt)

Input prompt to:
write a complete python function
based on the following description:

def count_upper(s):
    """
    Given a string s, count the number of uppercase vowels in even indices.
    
    For example:
    count_upper('aBCdEf') returns 1
    count_upper('abcdefg') returns 0
    count_upper('dBBE') returns 0
    """
.

with the following constraints:

Output only a complete and valid Python code for this function. 
Do not add more explanations or surrounding text and Do not change the provided function signature.
Wrap your output strictly between the markers:
<code>
... your code ...
</code>




In [8]:
def generate_response(input_prompt):
    messages = [
        {"role": "system", "content": "You are a useful assistant."},
        {"role": "user", "content": f"{input_prompt}"},
    ]

    outputs = pipeline(
        messages,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    response = outputs[0]["generated_text"][-1]['content']

    return response

In [9]:
output = generate_response(input_prompt)
print("Response:\n", output)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


Response:
 <code>
def count_upper(s):
    """
    Given a string s, count the number of uppercase vowels in even indices.
    
    For example:
    count_upper('aBCdEf') returns 1
    count_upper('abcdefg') returns 0
    count_upper('dBBE') returns 0
    """
    count = 0
    for i, char in enumerate(s):
        if i % 2 == 0 and char in 'AEIOU':
            count += 1
    return count
</code>


We can extract the complete code

In [10]:
completed_code = extract_function(output)
print(f"The complete code:\n")
print(completed_code)

The complete code:

def count_upper(s):
    """
    Given a string s, count the number of uppercase vowels in even indices.
    
    For example:
    count_upper('aBCdEf') returns 1
    count_upper('abcdefg') returns 0
    count_upper('dBBE') returns 0
    """
    count = 0
    for i, char in enumerate(s):
        if i % 2 == 0 and char in 'AEIOU':
            count += 1
    return count


## 2.2. Evaluate the generated code

In [11]:
import builtins
import typing

def create_namespace():
    ns = {}

    # 1. Standard builtins (print, len, etc.)
    ns.update({k: getattr(builtins, k) for k in dir(builtins)})

    # 2. Install common typing names (List, Optional, etc.)
    for name in typing.__all__:
        ns[name] = getattr(typing, name)

    # 3. (Optional) Add math, random, itertools, etc.
    import math, random, itertools, statistics
    ns.update({
        'math': math,
        'random': random,
        'itertools': itertools,
        'statistics': statistics,
    })

    return ns

In [12]:
def evaluate_asserts(generated_code: str, test_code: str, entry_point: str):
    # ns = {}
    ns = create_namespace()
    
    # 1. Exec both code strings
    exec(generated_code, ns)
    exec(test_code, ns)

    candidate = ns[entry_point]     # the model's function
    check_fn = ns["check"]          # original check() function
    
    # 2. Parse the test code AST
    tree = ast.parse(test_code)

    # 3. Find the check() function body
    check_body = None
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name == "check":
            check_body = node.body
            break

    if check_body is None:
        raise ValueError("check() function not found.")
    
    # 4. Evaluate each assert individually
    results = []
    for idx, stmt in enumerate(check_body):
        if isinstance(stmt, ast.Assert):
            # Convert AST back to executable code
            code = compile(ast.Module([stmt], type_ignores=[]), "<assert>", "exec")
            try:
                exec(code, {**ns, "candidate": candidate})
                results.append(("pass", None))
            except Exception as e:
                results.append(("fail", repr(e)))

    # 5. Compute pass percentage
    total = len(results)
    passed = sum(1 for r, _ in results if r == "pass")
    percentage = passed / total if total > 0 else 0.0

    return {
        "total_asserts": total,
        "passed": passed,
        "percentage": percentage,
        "detail": results
    }

In [13]:
result = evaluate_asserts(completed_code, test_case, entry_point)
print(result)

{'total_asserts': 8, 'passed': 8, 'percentage': 1.0, 'detail': [('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None)]}


# 3. Run through all sample

In [14]:
list_df = []

for idx in range(df.shape[0]):
    try:
        # 1. Prepare input prompt
        code_description = df.loc[idx, "prompt"]
        test_case = df.loc[idx, "test"]
        entry_point = df.loc[idx, "entry_point"]

        input_prompt = f"""write a complete python function
        based on the following description:\n{code_description}.\n
        with the following constraints:\n{constraints}
        """

        output = generate_response(input_prompt)
        completed_code = extract_function(output)
        
        # 3. Evaluate the generated code
        result = evaluate_asserts(completed_code, test_case, entry_point)
        total_asserts = result["total_asserts"]
        passed_asserts = result["passed"]
        percentage = result["percentage"]
        
        list_df.append({
            "description": code_description,
            "generated_code": completed_code,
            "test_case": test_case,
            "entry_point": entry_point,
            "total_asserts": total_asserts,
            "passed_asserts": passed_asserts,
            "percentage": percentage,
        })
    except Exception as e:
        print(f"Error at idx={idx}: {e}")
        continue

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id

In [15]:
output_df = pd.DataFrame(list_df)
print(f'Output dataframe shape: {output_df.shape}')
output_df.sample()

Output dataframe shape: (164, 7)


,description,generated_code,test_case,entry_point,total_asserts,passed_asserts,percentage
15,"\n\ndef string_sequence(n: int) -> str:\n ""...","def string_sequence(n: int) -> str:\n """""" R...","\n\nMETADATA = {\n 'author': 'jt',\n 'da...",string_sequence,3,1,0.333333


In [16]:
# Save to CSV
output_df.to_csv("data/generated/human_eval_generated_llama.csv", index=False)

## 3.1. Check generated code

In [17]:
average_percentage = output_df["percentage"].mean()
print(f"Average pass percentage over all samples: {average_percentage:.2%}")    

Average pass percentage over all samples: 83.05%
